# Module 9: Latent Diffusion & Advanced Topics

This module covers how diffusion scales to high-resolution images and the architectural ideas behind systems like Stable Diffusion, DALL-E 3, and Sora.

**Core idea:** Latent diffusion runs the noising process on a compressed representation instead of raw pixels, making it much faster.

**Learning Objectives**
- Understand why pixel-space diffusion hits a wall at higher resolutions
- Implement a convolutional VAE and the ELBO loss
- Build a latent diffusion pipeline: VAE encode → diffuse in latent space → decode
- Map the Stable Diffusion architecture (VAE + U-Net + text encoder)
- Compare three prediction targets: noise, clean image, and velocity
- Understand noise schedule improvements: cosine, offset noise, zero terminal SNR
- Implement a minimal Diffusion Transformer (DiT) with adaLN-Zero
- Implement flow matching as a simpler alternative to DDPM

**Estimated time:** 4-5 hours

**Key references:**
- [Latent Diffusion (Rombach et al. 2022)](https://arxiv.org/abs/2112.10752)
- [VAE (Kingma & Welling 2013)](https://arxiv.org/abs/1312.6114)
- [DiT (Peebles & Xie 2023)](https://arxiv.org/abs/2212.09748)
- [Flow Matching (Lipman et al. 2023)](https://arxiv.org/abs/2210.02747)
- [Rectified Flow (Liu et al. 2023)](https://arxiv.org/abs/2209.03003)
- [EDM (Karras et al. 2022)](https://arxiv.org/abs/2206.00364)

In [ ]:
import sys
import math
import time
from typing import Optional, Dict, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

sys.path.insert(0, '.')
from utils.schedule import cosine_schedule, linear_schedule, get_schedule
from utils.visualization import show_images, denormalize, plot_loss_curve, set_style
from utils.data import get_mnist_dataloader, get_device

torch.manual_seed(42)
device = get_device()
print(f"Using device: {device}")
set_style()

---

## 9.1 — The Pixel-Space Bottleneck

Running diffusion directly on pixels gets expensive fast. Consider:
- A **28x28** MNIST image has 784 dimensions
- A **256x256** RGB image has 196,608 dimensions
- A **512x512** RGB image has 786,432 dimensions

The U-Net must process all those dimensions at every denoising step, and you need hundreds of steps. Memory and compute scale roughly with the square of spatial resolution.

**The fix:** compress the image first with a VAE, then run diffusion in the smaller latent space.

### Tensor shape walkthrough

| Stage | Shape | Dimensions |
|-------|-------|-----------|
| Input image | `(B, 3, 256, 256)` | 196,608 |
| VAE encoder output | `(B, 4, 32, 32)` | 4,096 |
| Diffusion operates here | `(B, 4, 32, 32)` | 4,096 |
| VAE decoder output | `(B, 3, 256, 256)` | 196,608 |

That is a **48x** compression. The diffusion model never sees raw pixels.

### Worked Example: Profiling the Resolution Bottleneck

The cell below measures how U-Net forward pass time grows with image resolution.

In [ ]:
class SimpleConvBlock(nn.Module):
    """A minimal conv block to simulate U-Net processing at various resolutions."""
    def __init__(self, channels: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, channels, 3, padding=1),    # (B, 64, H, W)
            nn.GroupNorm(8, channels),
            nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),  # (B, 64, H, W)
            nn.GroupNorm(8, channels),
            nn.SiLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.net(x)  # (B, C, H, W)
        B, C, H, W = h.shape
        flat = h.view(B, C, H * W)        # (B, C, H*W)
        attn = torch.bmm(flat.transpose(1, 2), flat)  # (B, H*W, H*W) — quadratic!
        attn = F.softmax(attn / math.sqrt(C), dim=-1)
        h = torch.bmm(flat, attn.transpose(1, 2)).view(B, C, H, W)
        return h


# Profile memory at different resolutions
resolutions = [32, 64, 128, 256]
results = []

for res in resolutions:
    model = SimpleConvBlock(channels=64)
    x = torch.randn(1, 3, res, res)
    pixels = 3 * res * res
    attn_tokens = res * res
    attn_matrix_size = attn_tokens ** 2

    start = time.time()
    try:
        with torch.no_grad():
            _ = model(x)
        elapsed = time.time() - start
        oom = False
    except RuntimeError:
        elapsed = float('inf')
        oom = True

    results.append({
        'res': res, 'pixels': pixels,
        'attn_tokens': attn_tokens, 'attn_matrix': attn_matrix_size,
        'time_ms': elapsed * 1000, 'oom': oom
    })

print(f"{'Res':>6s} {'Pixels':>10s} {'Attn tokens':>12s} {'Attn matrix':>14s} {'Time (ms)':>10s}")
print("-" * 60)
for r in results:
    t_str = 'OOM' if r['oom'] else f"{r['time_ms']:.1f}"
    print(f"{r['res']:>6d} {r['pixels']:>10,d} {r['attn_tokens']:>12,d} {r['attn_matrix']:>14,d} {t_str:>10s}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
valid = [r for r in results if not r['oom']]
ax1.plot([r['res'] for r in valid], [r['attn_matrix'] for r in valid], 'o-', color='crimson')
ax1.set_xlabel('Resolution'); ax1.set_ylabel('Attention matrix elements')
ax1.set_title('Attention: O(n²) in tokens, O(res⁴) for images'); ax1.set_yscale('log')

ax2.plot([r['res'] for r in valid], [r['time_ms'] for r in valid], 's-', color='steelblue')
ax2.set_xlabel('Resolution'); ax2.set_ylabel('Forward pass time (ms)')
ax2.set_title('Compute time vs resolution')
plt.tight_layout(); plt.show()

print(f"\n256×256 attention matrix: {256**4:,} elements — this is why we need latent diffusion!")

---

## 9.2 — Variational Autoencoders (VAEs)

A **VAE** learns to compress images into a low-dimensional latent space and reconstruct them.

**Encoder** maps an image to a distribution in latent space:
$$q_\phi(z|x) = \mathcal{N}(\mu_\phi(x),\; \sigma^2_\phi(x))$$

**Decoder** reconstructs the image from a latent sample:
$$p_\theta(x|z)$$

**ELBO loss** (what we maximize):
$$\mathcal{L} = \underbrace{\mathbb{E}_{q(z|x)}[\log p(x|z)]}_{\text{reconstruction}} - \underbrace{D_\text{KL}(q(z|x) \| p(z))}_{\text{regularization}}$$

- The **reconstruction** term pushes the decoder to faithfully reproduce the input
- The **KL** term keeps the latent distribution close to a standard Gaussian $\mathcal{N}(0, I)$, which is what we sample from at generation time

**Reparameterization trick:** To backprop through sampling, write $z = \mu + \sigma \odot \epsilon$ where $\epsilon \sim \mathcal{N}(0, I)$. The randomness is moved into $\epsilon$, making $z$ a deterministic function of the parameters.

### Tensor shapes through the VAE

For our MNIST example with `latent_dim=32`:

| Layer | Output shape |
|-------|-------------|
| Input | `(B, 1, 28, 28)` |
| Conv encoder | `(B, 64, 7, 7)` |
| Flatten + Linear | `(B, 32)` for $\mu$, `(B, 32)` for $\log\sigma^2$ |
| Sample $z$ | `(B, 32)` |
| Linear + Unflatten | `(B, 64, 7, 7)` |
| ConvTranspose decoder | `(B, 1, 28, 28)` |

### Worked Example: Convolutional VAE on MNIST

In [ ]:
class ConvVAE(nn.Module):
    """Convolutional VAE for 28x28 grayscale images.

    Encoder: 1x28x28 -> 32x14x14 -> 64x7x7 -> flatten -> (mu, log_var) of dim latent_dim
    Decoder: latent_dim -> 64x7x7 -> 32x14x14 -> 1x28x28
    """
    def __init__(self, latent_dim: int = 32):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),  # (B, 32, 14, 14)
            nn.GroupNorm(min(32, 32), 32),  # GroupNorm: consistent with diffusion best practices
            nn.SiLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), # (B, 64, 7, 7)
            nn.GroupNorm(min(32, 64), 64),
            nn.SiLU(),
        )
        self.fc_mu = nn.Linear(64 * 7 * 7, latent_dim)      # (B, latent_dim)
        self.fc_logvar = nn.Linear(64 * 7 * 7, latent_dim)   # (B, latent_dim)

        # Decoder
        self.fc_decode = nn.Linear(latent_dim, 64 * 7 * 7)   # (B, 64*7*7)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),  # (B, 32, 14, 14)
            nn.GroupNorm(min(32, 32), 32),  # GroupNorm: consistent with diffusion best practices
            nn.SiLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1),   # (B, 1, 28, 28)
            nn.Tanh(),  # Output in [-1, 1] to match data normalization
        )

    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Encode image to latent distribution parameters."""
        h = self.encoder(x)             # (B, 64, 7, 7)
        h = h.view(h.size(0), -1)       # (B, 64*7*7)
        return self.fc_mu(h), self.fc_logvar(h)  # (B, latent_dim), (B, latent_dim)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """Sample z = mu + sigma * epsilon using the reparameterization trick."""
        std = torch.exp(0.5 * logvar)   # (B, latent_dim)
        eps = torch.randn_like(std)     # (B, latent_dim)
        return mu + std * eps           # (B, latent_dim)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """Decode latent vector to image."""
        h = self.fc_decode(z)           # (B, 64*7*7)
        h = h.view(-1, 64, 7, 7)       # (B, 64, 7, 7)
        return self.decoder(h)          # (B, 1, 28, 28)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Full forward: encode -> sample -> decode."""
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(
    x: torch.Tensor, x_recon: torch.Tensor,
    mu: torch.Tensor, logvar: torch.Tensor,
    kl_weight: float = 1.0,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute VAE loss = reconstruction + KL divergence."""
    recon = F.mse_loss(x_recon, x, reduction='mean')
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + kl_weight * kl, recon, kl


# Quick shape check
vae = ConvVAE(latent_dim=32).to(device)
test_x = torch.randn(4, 1, 28, 28, device=device)
recon, mu, logvar = vae(test_x)
print(f"Input shape:  {test_x.shape}")
print(f"Latent shape: {mu.shape}")
print(f"Recon shape:  {recon.shape}")
print(f"Compression:  {28*28}/{mu.shape[1]} = {28*28/mu.shape[1]:.0f}x")

In [ ]:
# Train the VAE on MNIST
torch.manual_seed(42)

vae = ConvVAE(latent_dim=32).to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)
dataloader = get_mnist_dataloader(batch_size=128)

num_epochs = 10
losses_hist = {'total': [], 'recon': [], 'kl': []}

for epoch in range(num_epochs):
    epoch_loss = 0.0
    for images, _ in dataloader:
        images = images.to(device)  # (B, 1, 28, 28)
        x_recon, mu, logvar = vae(images)
        loss, recon, kl = vae_loss(images, x_recon, mu, logvar, kl_weight=0.5)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(dataloader)
    losses_hist['total'].append(avg_loss)
    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

print("VAE training complete!")

In [ ]:
# Visualize VAE reconstructions and latent interpolation
vae.eval()
test_batch, _ = next(iter(dataloader))
test_batch = test_batch[:8].to(device)

with torch.no_grad():
    recon, _, _ = vae(test_batch)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(denormalize(test_batch[i]).cpu().squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(denormalize(recon[i]).cpu().squeeze(), cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('Recon', fontsize=12)
plt.suptitle('VAE Reconstructions', fontsize=14)
plt.tight_layout(); plt.show()

# Latent space interpolation
with torch.no_grad():
    mu1, _ = vae.encode(test_batch[0:1])
    mu2, _ = vae.encode(test_batch[4:5])
    n_steps = 10
    alphas_interp = torch.linspace(0, 1, n_steps, device=device)
    interp_z = torch.stack([mu1 * (1 - a) + mu2 * a for a in alphas_interp]).squeeze(1)
    interp_imgs = vae.decode(interp_z)

fig, axes = plt.subplots(1, n_steps, figsize=(20, 2))
for i in range(n_steps):
    axes[i].imshow(denormalize(interp_imgs[i]).cpu().squeeze(), cmap='gray')
    axes[i].axis('off')
    axes[i].set_title(f'{alphas_interp[i]:.1f}')
plt.suptitle('Latent Space Interpolation', fontsize=14)
plt.tight_layout(); plt.show()

### Exercise 9.1: Explore VAE Latent Dimension

Train two VAEs with different latent dimensions (e.g., `latent_dim=8` and `latent_dim=64`) for 5 epochs each. Compare:

1. Reconstruction quality
2. Samples from the prior $z \sim \mathcal{N}(0, I)$ decoded to images

You should see a tradeoff: lower dimensions give blurrier reconstructions but a smoother latent space, while higher dimensions give sharper reconstructions but noisier prior samples.

In [ ]:
# Exercise 9.1 — YOUR CODE HERE

# ========================== END YOUR CODE ==========================

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
torch.manual_seed(42)

fig, axes = plt.subplots(3, 8, figsize=(16, 6))
for row, ldim in enumerate([8, 32, 64]):
    vae_test = ConvVAE(latent_dim=ldim).to(device)
    opt = torch.optim.Adam(vae_test.parameters(), lr=1e-3)
    for epoch in range(5):
        for imgs, _ in dataloader:
            imgs = imgs.to(device)
            recon, mu, logvar = vae_test(imgs)
            loss, _, _ = vae_loss(imgs, recon, mu, logvar, kl_weight=0.5)
            opt.zero_grad(); loss.backward(); opt.step()
    vae_test.eval()
    with torch.no_grad():
        z_random = torch.randn(8, ldim, device=device)
        samples = vae_test.decode(z_random)
    for i in range(8):
        axes[row, i].imshow(denormalize(samples[i]).cpu().squeeze(), cmap='gray')
        axes[row, i].axis('off')
    axes[row, 0].set_ylabel(f'dim={ldim}', fontsize=12)
plt.suptitle('Random Samples from Prior at Different Latent Dims', fontsize=14)
plt.tight_layout(); plt.show()
print("Lower dim -> blurrier but smoother latent space. Higher dim -> sharper but harder to sample.")

---

## 9.3 — Latent Diffusion

The full pipeline:
1. **Train a VAE** on your image dataset
2. **Encode** all training images: `z = encoder(x)` with shape `(B, latent_dim)`
3. **Train a diffusion model** on the latent codes (same DDPM math, just on `z` instead of `x`)
4. **Sample:** generate `z` via reverse diffusion, then decode: `x = decoder(z)`

The diffusion model is much smaller and faster because it operates on compressed representations.

### Why normalize latents?

VAE latent codes may have non-unit variance. Before training diffusion, normalize them to roughly $\mathcal{N}(0, I)$ so the noise schedule assumptions hold:
$$z_{\text{norm}} = \frac{z - \mu_z}{\sigma_z}$$

At sampling time, undo the normalization before decoding.

### Worked Example: Latent Diffusion on MNIST

We encode MNIST to 32-d latents, train an MLP denoiser on those latents, then decode samples back to images.

In [ ]:
# Step 1: Encode all MNIST training images to latent space
vae.eval()
all_latents = []
with torch.no_grad():
    for images, _ in tqdm(dataloader, desc="Encoding MNIST"):
        images = images.to(device)
        mu, _ = vae.encode(images)  # (B, 32) — use mean, no sampling noise
        all_latents.append(mu.cpu())

latent_dataset = torch.cat(all_latents, dim=0)  # (60000, 32)
print(f"Latent dataset shape: {latent_dataset.shape}")
print(f"Latent stats: mean={latent_dataset.mean():.3f}, std={latent_dataset.std():.3f}")

# Normalize latents to roughly unit variance for diffusion
latent_mean = latent_dataset.mean(dim=0, keepdim=True)  # (1, 32)
latent_std = latent_dataset.std(dim=0, keepdim=True)    # (1, 32)
latent_dataset_norm = (latent_dataset - latent_mean) / (latent_std + 1e-6)
print(f"Normalized stats: mean={latent_dataset_norm.mean():.3f}, std={latent_dataset_norm.std():.3f}")

In [ ]:
class LatentDiffusionMLP(nn.Module):
    """Simple MLP for diffusion on 1D latent codes.

    Since our latents are flat vectors (not spatial), we use an MLP
    instead of a U-Net. Predicts noise given (z_t, t).
    """
    def __init__(self, latent_dim: int = 32, hidden_dim: int = 256, time_dim: int = 64):
        super().__init__()
        # A single nn.Linear(1, time_dim) is insufficient for time conditioning —
        # the model needs nonlinear features of t to distinguish noise levels.
        # A small MLP (or sinusoidal embedding) gives much better results.
        self.time_embed = nn.Sequential(
            nn.Linear(1, time_dim),       # (B, time_dim)
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        self.net = nn.Sequential(
            nn.Linear(latent_dim + time_dim, hidden_dim),  # (B, hidden_dim)
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, latent_dim),  # (B, latent_dim)
        )

    def forward(self, z_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """Predict noise given noisy latent and timestep.

        Args:
            z_t: (B, latent_dim) noisy latent
            t: (B,) or (B, 1) timestep in [0, 1]
        """
        if t.dim() == 1:
            t = t.unsqueeze(1)
        t_emb = self.time_embed(t)           # (B, time_dim)
        x = torch.cat([z_t, t_emb], dim=1)  # (B, latent_dim + time_dim)
        return self.net(x)                   # (B, latent_dim)"

In [ ]:
# Step 2: Train diffusion model on latent codes
torch.manual_seed(42)

T = 1000
schedule = cosine_schedule(T)
sqrt_alpha_bar = schedule['sqrt_alphas_cumprod'].to(device)
sqrt_one_minus_alpha_bar = schedule['sqrt_one_minus_alphas_cumprod'].to(device)

latent_model = LatentDiffusionMLP(latent_dim=32, hidden_dim=256).to(device)
optimizer = torch.optim.Adam(latent_model.parameters(), lr=1e-3)

latent_loader = DataLoader(TensorDataset(latent_dataset_norm), batch_size=256, shuffle=True)

num_epochs = 30
loss_history = []

for epoch in range(num_epochs):
    epoch_loss = 0.0
    for (z_0,) in latent_loader:
        z_0 = z_0.to(device)  # (B, 32)
        B = z_0.shape[0]
        t_idx = torch.randint(0, T, (B,), device=device)
        noise = torch.randn_like(z_0)  # (B, 32)
        z_t = sqrt_alpha_bar[t_idx].unsqueeze(1) * z_0 + \
              sqrt_one_minus_alpha_bar[t_idx].unsqueeze(1) * noise  # (B, 32)
        t_norm = t_idx.float() / T
        noise_pred = latent_model(z_t, t_norm)
        loss = F.mse_loss(noise_pred, noise)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(latent_loader)
    loss_history.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

plot_loss_curve(loss_history, title='Latent Diffusion Training Loss')

### Exercise 9.2: Compare Pixel-Space vs Latent Diffusion

Time how long it takes to train a pixel-space MLP diffusion model on flattened MNIST (784 dims) vs a latent-space MLP (32 dims) for 10 epochs each. Compare training time and final loss.

In [ ]:
# Exercise 9.2 — YOUR CODE HERE

# ========================== END YOUR CODE ==========================

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
torch.manual_seed(42)

# Pixel-space: flatten MNIST
all_pixels = []
for images, _ in dataloader:
    all_pixels.append(images.view(images.size(0), -1))
pixel_dataset = torch.cat(all_pixels, dim=0)  # (60000, 784)
pixel_loader = DataLoader(TensorDataset(pixel_dataset), batch_size=256, shuffle=True)

for name, dim, loader in [('Pixel (784d)', 784, pixel_loader), ('Latent (32d)', 32, latent_loader)]:
    model = LatentDiffusionMLP(latent_dim=dim, hidden_dim=min(dim * 4, 512)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    start = time.time()
    final_loss = 0
    for epoch in range(10):
        for (batch,) in loader:
            batch = batch.to(device)
            B = batch.shape[0]
            t_idx = torch.randint(0, T, (B,), device=device)
            noise = torch.randn_like(batch)
            noisy = sqrt_alpha_bar[t_idx].unsqueeze(1) * batch + \
                    sqrt_one_minus_alpha_bar[t_idx].unsqueeze(1) * noise
            pred = model(noisy, t_idx.float() / T)
            loss = F.mse_loss(pred, noise)
            opt.zero_grad(); loss.backward(); opt.step()
            final_loss = loss.item()
    elapsed = time.time() - start
    print(f"{name}: {elapsed:.1f}s for 10 epochs, final loss={final_loss:.4f}")

---

## 9.4 — The Stable Diffusion Architecture

Stable Diffusion has three components:

1. **VAE** (autoencoder)
   - Encoder: `(B, 3, 512, 512)` → `(B, 4, 64, 64)`
   - Decoder: `(B, 4, 64, 64)` → `(B, 3, 512, 512)`
   - Trained separately, frozen during diffusion training

2. **U-Net** (denoiser, operates in latent space)
   - Input: noisy latent `(B, 4, 64, 64)` + timestep + text embedding
   - Output: predicted noise `(B, 4, 64, 64)`
   - Cross-attention layers attend to text embeddings

3. **Text encoder** (CLIP or T5)
   - Maps text prompt to sequence of embeddings `(B, seq_len, 768)`
   - These embeddings are injected into the U-Net via cross-attention

**Inference pipeline:** text → text encoder → embeddings → U-Net denoises latent (with CFG) → VAE decoder → image

---

## 9.5 — Prediction Parameterizations

The denoising network can predict different targets. All three are mathematically equivalent (you can convert between them), but they behave differently during training:

- **$\epsilon$-prediction:** predict the noise added at timestep $t$. Standard DDPM. Works well overall but has high variance near $t=0$ (dividing by small $\sqrt{\bar\alpha_t}$).

- **$x_0$-prediction:** predict the clean data directly. Constant loss magnitude, but struggles at high noise levels where the signal is nearly destroyed.

- **$v$-prediction:** predict $v = \sqrt{\bar\alpha_t}\,\epsilon - \sqrt{1-\bar\alpha_t}\,x_0$. Balances the loss across all timesteps. Used in Stable Diffusion 2.0+.

### Conversion formulas

Given $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$:

| From | To $x_0$ | To $\epsilon$ |
|------|----------|---------------|
| $\epsilon$ | $(x_t - \sqrt{1-\bar\alpha_t}\,\epsilon) / \sqrt{\bar\alpha_t}$ | identity |
| $x_0$ | identity | $(x_t - \sqrt{\bar\alpha_t}\,x_0) / \sqrt{1-\bar\alpha_t}$ |
| $v$ | $\sqrt{\bar\alpha_t}\,x_t - \sqrt{1-\bar\alpha_t}\,v$ | $\sqrt{1-\bar\alpha_t}\,x_t + \sqrt{\bar\alpha_t}\,v$ |

### Worked Example: Conversion Functions and Equivalence

In [ ]:
def compute_v_target(x_0, epsilon, sqrt_alpha_bar, sqrt_one_minus_alpha_bar):
    """v = sqrt(alpha_bar) * eps - sqrt(1 - alpha_bar) * x_0"""
    return sqrt_alpha_bar * epsilon - sqrt_one_minus_alpha_bar * x_0

def eps_to_x0(x_t, eps, sab, somab):
    """Convert eps-prediction to x0-prediction."""
    return (x_t - somab * eps) / sab

def x0_to_eps(x_t, x0, sab, somab):
    """Convert x0-prediction to eps-prediction."""
    return (x_t - sab * x0) / somab

def v_to_x0(x_t, v, sab, somab):
    """Convert v-prediction to x0-prediction."""
    return sab * x_t - somab * v

def v_to_eps(x_t, v, sab, somab):
    """Convert v-prediction to eps-prediction."""
    return somab * x_t + sab * v


# Demonstrate equivalence
torch.manual_seed(42)
x_0 = torch.randn(4, 32)
eps = torch.randn(4, 32)
t = 500
sab = schedule['sqrt_alphas_cumprod'][t]
somab = schedule['sqrt_one_minus_alphas_cumprod'][t]
x_t = sab * x_0 + somab * eps
v = compute_v_target(x_0, eps, sab, somab)

# Verify round-trip conversions
print(f"x0 from eps: max error = {(eps_to_x0(x_t, eps, sab, somab) - x_0).abs().max():.2e}")
print(f"x0 from v:   max error = {(v_to_x0(x_t, v, sab, somab) - x_0).abs().max():.2e}")
print(f"eps from v:  max error = {(v_to_eps(x_t, v, sab, somab) - eps).abs().max():.2e}")
print("\nAll three parameterizations are mathematically equivalent!")

### Exercise 9.3: Train All Three Prediction Modes

Modify the `LatentDiffusionMLP` training loop to support all three parameterizations. Train for 15 epochs each and compare final losses.

In [ ]:
# Exercise 9.3 — YOUR CODE HERE

# ========================== END YOUR CODE ==========================

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
torch.manual_seed(42)
results_pred = {}
for mode in ['eps', 'x0', 'v']:
    model = LatentDiffusionMLP(latent_dim=32, hidden_dim=256).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    losses = []
    for epoch in range(15):
        epoch_loss = 0
        for (z_0,) in latent_loader:
            z_0 = z_0.to(device)
            B = z_0.shape[0]
            t_idx = torch.randint(0, T, (B,), device=device)
            noise = torch.randn_like(z_0)
            sab_t = sqrt_alpha_bar[t_idx].unsqueeze(1)
            somab_t = sqrt_one_minus_alpha_bar[t_idx].unsqueeze(1)
            z_t = sab_t * z_0 + somab_t * noise
            pred = model(z_t, t_idx.float() / T)
            if mode == 'eps':
                target = noise
            elif mode == 'x0':
                target = z_0
            else:
                target = sab_t * noise - somab_t * z_0
            loss = F.mse_loss(pred, target)
            opt.zero_grad(); loss.backward(); opt.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(latent_loader))
    results_pred[mode] = losses
    print(f"{mode}-prediction: final loss = {losses[-1]:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
for mode, losses in results_pred.items():
    ax.plot(losses, label=f'{mode}-prediction')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Training Loss by Prediction Parameterization')
ax.legend(); plt.tight_layout(); plt.show()

---

## 9.6 — Noise Schedule Improvements

### Cosine schedule (Improved DDPM)

Smoother SNR transition than linear:
$$\bar{\alpha}_t = \frac{f(t)}{f(0)}, \quad f(t) = \cos\left(\frac{t/T + s}{1 + s} \cdot \frac{\pi}{2}\right)^2$$

### Offset noise

Standard Gaussian noise has zero mean, so the model can never generate images with uniform brightness. **Offset noise** adds a per-channel bias:
$$\epsilon = \epsilon_{\text{standard}} + \delta \cdot \epsilon_{\text{offset}}, \quad \epsilon_{\text{offset}} \in \mathbb{R}^{B \times C \times 1 \times 1}$$

### Zero terminal SNR

Many schedules never reach $\bar{\alpha}_T = 0$, creating a train-inference mismatch. The fix rescales:
$$\bar{\alpha}_t' = \frac{\bar{\alpha}_t - \bar{\alpha}_T}{1 - \bar{\alpha}_T}$$

### Log-SNR linear schedule

Linear in log-SNR space gives equal difficulty across timesteps.

### Worked Example: Schedule Comparison

In [ ]:
T_sched = 1000
lin_sched = linear_schedule(T_sched)
cos_sched = cosine_schedule(T_sched)

# Log-SNR linear schedule
def log_snr_linear_schedule(T, snr_min=0.001, snr_max=1000.0):
    """Schedule linear in log-SNR space."""
    log_snr = torch.linspace(math.log(snr_max), math.log(snr_min), T)
    snr = log_snr.exp()
    return snr / (1 + snr)  # SNR = a_bar/(1-a_bar) -> a_bar = SNR/(1+SNR)

log_snr_abar = log_snr_linear_schedule(T_sched)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
t_range = np.arange(T_sched)

axes[0].plot(t_range, lin_sched['alphas_cumprod'].numpy(), label='Linear', alpha=0.8)
axes[0].plot(t_range, cos_sched['alphas_cumprod'].numpy(), label='Cosine', alpha=0.8)
axes[0].plot(t_range, log_snr_abar.numpy(), label='Log-SNR linear', alpha=0.8)
axes[0].set_xlabel('Timestep'); axes[0].set_ylabel('alpha_bar_t')
axes[0].set_title('Cumulative Signal Retention'); axes[0].legend()

for sched, label in [(lin_sched, 'Linear'), (cos_sched, 'Cosine')]:
    abar = sched['alphas_cumprod'].clamp(min=1e-8)
    snr = abar / (1 - abar).clamp(min=1e-8)
    axes[1].plot(t_range, snr.log().numpy(), label=label, alpha=0.8)
snr_ls = log_snr_abar / (1 - log_snr_abar).clamp(min=1e-8)
axes[1].plot(t_range, snr_ls.log().numpy(), label='Log-SNR linear', alpha=0.8)
axes[1].set_xlabel('Timestep'); axes[1].set_ylabel('log SNR')
axes[1].set_title('Log Signal-to-Noise Ratio'); axes[1].legend()

axes[2].bar(['Linear', 'Cosine', 'Log-SNR'],
    [lin_sched['alphas_cumprod'][-1].item(), cos_sched['alphas_cumprod'][-1].item(), log_snr_abar[-1].item()],
    color=['steelblue', 'coral', 'forestgreen'])
axes[2].set_ylabel('alpha_bar_T (should be ~0)')
axes[2].set_title('Terminal SNR (lower = better)'); axes[2].set_yscale('log')
plt.tight_layout(); plt.show()

### Exercise 9.4: Zero Terminal SNR Schedule

Implement a rescaled cosine schedule that enforces zero terminal SNR:
$$\bar{\alpha}_t' = \frac{\bar{\alpha}_t - \bar{\alpha}_T}{1 - \bar{\alpha}_T}$$

Plot original vs rescaled and verify $\bar{\alpha}_T' = 0$.

In [ ]:
# Exercise 9.4 — YOUR CODE HERE

# ========================== END YOUR CODE ==========================

In [ ]:
# ===================== SOLUTION: Exercise 9.4 =====================

def enforce_zero_terminal_snr(schedule_dict: dict) -> dict:
    """Rescale a noise schedule so that the final timestep has zero terminal SNR.

    Standard schedules have alphas_cumprod[-1] > 0, meaning the model never sees
    pure noise during training. Zero terminal SNR fixes this by rescaling:

        alpha_bar'_t = (alpha_bar_t - alpha_bar_T) / (1 - alpha_bar_T)

    This maps alpha_bar from [alpha_bar_T, 1] to [0, 1], ensuring:
    - alpha_bar'_0 stays at 1 (clean image at t=0)
    - alpha_bar'_T becomes exactly 0 (pure noise at t=T)

    Args:
        schedule_dict: dict with 'alphas_cumprod' key (and optionally others)
    Returns:
        new dict with rescaled schedule tensors
    """
    alphas_cumprod = schedule_dict['alphas_cumprod']
    alpha_T = alphas_cumprod[-1]

    # Rescale so final value is exactly 0
    alphas_cumprod_rescaled = (alphas_cumprod - alpha_T) / (1 - alpha_T)

    # Clamp to avoid numerical issues at boundaries
    alphas_cumprod_rescaled = alphas_cumprod_rescaled.clamp(min=0.0, max=1.0)

    # Recompute betas from rescaled alphas_cumprod
    # beta_t = 1 - alpha_bar_t / alpha_bar_{t-1}
    alphas = alphas_cumprod_rescaled[1:] / alphas_cumprod_rescaled[:-1]
    alphas = torch.cat([alphas_cumprod_rescaled[:1], alphas])
    betas = 1.0 - alphas
    betas = betas.clamp(min=0.0, max=0.999)

    # Recompute derived quantities
    sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod_rescaled)
    sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod_rescaled)

    return {
        'betas': betas,
        'alphas_cumprod': alphas_cumprod_rescaled,
        'sqrt_alphas_cumprod': sqrt_alphas_cumprod,
        'sqrt_one_minus_alphas_cumprod': sqrt_one_minus_alphas_cumprod,
    }


# Compare original vs zero-terminal-SNR cosine schedule
T_compare = 1000
original_schedule = cosine_schedule(T_compare)
rescaled_schedule = enforce_zero_terminal_snr(original_schedule)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
timesteps = torch.arange(T_compare)

# Plot alphas_cumprod
axes[0].plot(timesteps.numpy(), original_schedule['alphas_cumprod'].numpy(),
             label='Original cosine', alpha=0.8)
axes[0].plot(timesteps.numpy(), rescaled_schedule['alphas_cumprod'].numpy(),
             label='Zero terminal SNR', alpha=0.8, linestyle='--')
axes[0].set_xlabel('Timestep')
axes[0].set_ylabel(r'$\bar{\alpha}_t$')
axes[0].set_title(r'$\bar{\alpha}_t$ comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot SNR = alpha_bar / (1 - alpha_bar) in log scale
snr_orig = original_schedule['alphas_cumprod'] / (1 - original_schedule['alphas_cumprod'] + 1e-8)
snr_rescaled = rescaled_schedule['alphas_cumprod'] / (1 - rescaled_schedule['alphas_cumprod'] + 1e-8)
axes[1].semilogy(timesteps.numpy(), snr_orig.numpy(), label='Original cosine', alpha=0.8)
axes[1].semilogy(timesteps.numpy(), snr_rescaled.numpy(), label='Zero terminal SNR', alpha=0.8, linestyle='--')
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('SNR (log scale)')
axes[1].set_title('Signal-to-Noise Ratio')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Verify terminal values
print(f"Original  alpha_bar[T-1] = {original_schedule['alphas_cumprod'][-1]:.6f}")
print(f"Rescaled  alpha_bar[T-1] = {rescaled_schedule['alphas_cumprod'][-1]:.6f}")
print(f"Rescaled  alpha_bar[0]   = {rescaled_schedule['alphas_cumprod'][0]:.6f}")

# ===================== END SOLUTION =====================

---

## 9.7 — DiT: Diffusion Transformer

The **Diffusion Transformer (DiT)** replaces the U-Net with a vision transformer. It powers Sora and Stable Diffusion 3.

### Architecture

1. **Patchify:** split image into non-overlapping patches, project to embeddings
2. **Positional encoding:** add learned position embeddings
3. **Transformer blocks:** self-attention + FFN with **adaLN-Zero** conditioning
4. **Unpatchify:** reshape back to spatial grid

### adaLN-Zero conditioning

Standard adaptive layer norm scales and shifts the normalized activations:
$$\text{adaLN}(h, c) = \gamma(c) \cdot \text{LayerNorm}(h) + \beta(c)$$

The "Zero" variant also predicts a gate $\alpha(c)$ initialized to zero, so each block starts as an identity function. This stabilizes early training.

### Why replace U-Net with a transformer?

- **Scaling:** transformers scale more predictably with compute
- **Simplicity:** no skip connections or encoder-decoder asymmetry
- **Flexibility:** handles variable-length sequences naturally

### Worked Example: Minimal DiT Implementation

In [ ]:
from utils.unet import SinusoidalTimestepEmbedding


class PatchEmbed(nn.Module):
    """Convert image into a sequence of patch embeddings.

    Splits the image into non-overlapping patches using a strided convolution,
    then projects each patch to the embedding dimension.

    For MNIST (28x28) with patch_size=7, we get (28/7)^2 = 16 patches.
    """

    def __init__(self, img_size: int = 28, patch_size: int = 7,
                 in_channels: int = 1, embed_dim: int = 128):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2  # 16 for MNIST
        self.proj = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: images, shape (B, C, H, W)
        Returns:
            patch embeddings, shape (B, num_patches, embed_dim)
        """
        x = self.proj(x)                    # (B, embed_dim, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)    # (B, num_patches, embed_dim)
        return x


class DiTBlock(nn.Module):
    """Transformer block with adaptive layer norm (adaLN-Zero).

    Instead of plain LayerNorm, we modulate the normalized activations using
    scale and shift parameters predicted from the conditioning signal.

    Each sub-layer (attention, MLP) gets its own (scale, shift, gate) triple,
    so we predict 6 values total from the conditioning vector.

    The "Zero" in adaLN-Zero means gates are initialized near zero, so the
    block starts as an identity function — this stabilizes early training.
    """

    def __init__(self, embed_dim: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim, elementwise_affine=False)
        self.attn = nn.MultiheadAttention(
            embed_dim, num_heads, batch_first=True
        )
        self.norm2 = nn.LayerNorm(embed_dim, elementwise_affine=False)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
        )
        # adaLN modulation: predict (scale1, shift1, gate1, scale2, shift2, gate2)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(embed_dim, 6 * embed_dim),
        )
        # Initialize so gates start near zero (identity-like block)
        nn.init.zeros_(self.adaLN_modulation[1].weight)
        nn.init.zeros_(self.adaLN_modulation[1].bias)

    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: token sequence, shape (B, N, embed_dim)
            cond: conditioning vector (time + class), shape (B, embed_dim)
        Returns:
            updated tokens, shape (B, N, embed_dim)
        """
        # Predict 6 modulation parameters from conditioning
        mod = self.adaLN_modulation(cond)             # (B, 6 * embed_dim)
        scale1, shift1, gate1, scale2, shift2, gate2 = mod.chunk(6, dim=-1)
        # Each is (B, embed_dim)

        # --- Attention sub-layer with adaLN ---
        h = self.norm1(x)                             # (B, N, embed_dim)
        h = h * (1 + scale1.unsqueeze(1)) + shift1.unsqueeze(1)
        h, _ = self.attn(h, h, h)                     # (B, N, embed_dim)
        x = x + gate1.unsqueeze(1) * h                # gated residual

        # --- MLP sub-layer with adaLN ---
        h = self.norm2(x)                             # (B, N, embed_dim)
        h = h * (1 + scale2.unsqueeze(1)) + shift2.unsqueeze(1)
        h = self.mlp(h)                               # (B, N, embed_dim)
        x = x + gate2.unsqueeze(1) * h                # gated residual

        return x


class DiT(nn.Module):
    """Minimal Diffusion Transformer for MNIST.

    Architecture overview:
    1. Patchify: split 28x28 image into 16 patches of 7x7
    2. Add learned positional embeddings to each patch token
    3. Condition on timestep + class via adaLN-Zero in each block
    4. Final linear layer reconstructs patch pixels, reshape to image

    This follows the DiT paper (Peebles & Xie 2023) but simplified for MNIST.
    """

    def __init__(self, img_size: int = 28, patch_size: int = 7,
                 in_channels: int = 1, embed_dim: int = 128,
                 depth: int = 4, num_heads: int = 4, num_classes: int = 10):
        super().__init__()
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.img_size = img_size
        num_patches = (img_size // patch_size) ** 2  # 16

        # --- Input processing ---
        self.patch_embed = PatchEmbed(img_size, patch_size, in_channels, embed_dim)
        self.pos_embed = nn.Parameter(
            torch.zeros(1, num_patches, embed_dim)
        )  # learned positional embedding, shape (1, 16, 128)

        # --- Conditioning ---
        self.time_embed = SinusoidalTimestepEmbedding(embed_dim)
        self.class_embed = nn.Embedding(num_classes + 1, embed_dim)  # +1 for null class (CFG)

        # --- Transformer blocks ---
        self.blocks = nn.ModuleList([
            DiTBlock(embed_dim, num_heads) for _ in range(depth)
        ])

        # --- Output: project tokens back to pixel patches ---
        self.final_norm = nn.LayerNorm(embed_dim, elementwise_affine=False)
        self.final_adaLN = nn.Sequential(
            nn.SiLU(),
            nn.Linear(embed_dim, 2 * embed_dim),  # scale and shift for final norm
        )
        self.final_linear = nn.Linear(
            embed_dim, patch_size * patch_size * in_channels
        )
        # Initialize final linear to zero so model starts predicting zero noise
        nn.init.zeros_(self.final_linear.weight)
        nn.init.zeros_(self.final_linear.bias)

        # Initialize positional embedding
        nn.init.normal_(self.pos_embed, std=0.02)

    def forward(self, x: torch.Tensor, t: torch.Tensor,
                class_label: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            x: noisy images, shape (B, 1, 28, 28)
            t: timesteps, shape (B,)
            class_label: class indices, shape (B,). Use num_classes for null token.
        Returns:
            predicted noise, shape (B, 1, 28, 28)
        """
        B = x.shape[0]

        # Patchify + add positional embeddings
        x = self.patch_embed(x)              # (B, 16, 128)
        x = x + self.pos_embed               # (B, 16, 128)

        # Build conditioning vector: time embedding + class embedding
        cond = self.time_embed(t)            # (B, 128)
        if class_label is not None:
            cond = cond + self.class_embed(class_label)  # (B, 128)

        # Pass through transformer blocks
        for block in self.blocks:
            x = block(x, cond)               # (B, 16, 128)

        # Final projection: adaLN then linear to patch pixels
        scale, shift = self.final_adaLN(cond).chunk(2, dim=-1)
        x = self.final_norm(x)              # (B, 16, 128)
        x = x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)
        x = self.final_linear(x)            # (B, 16, 49)  [49 = 7*7*1]

        # Reshape patches back to image
        P = self.patch_size
        grid_size = self.img_size // P       # 4
        # (B, 16, 49) → (B, 4, 4, 7, 7, 1)
        x = x.reshape(B, grid_size, grid_size, P, P, self.in_channels)
        # Rearrange to (B, C, H, W)
        x = x.permute(0, 5, 1, 3, 2, 4)     # (B, 1, 4, 7, 4, 7)
        x = x.reshape(B, self.in_channels, self.img_size, self.img_size)  # (B, 1, 28, 28)

        return x


# Quick test: verify shapes
device = get_device()
dit = DiT(embed_dim=128, depth=4, num_heads=4).to(device)
dummy_x = torch.randn(4, 1, 28, 28, device=device)
dummy_t = torch.randint(0, 1000, (4,), device=device)
dummy_labels = torch.randint(0, 10, (4,), device=device)

out = dit(dummy_x, dummy_t, dummy_labels)
print(f"Input shape:  {dummy_x.shape}")   # (4, 1, 28, 28)
print(f"Output shape: {out.shape}")        # (4, 1, 28, 28)
print(f"Parameters:   {sum(p.numel() for p in dit.parameters()):,}")


### Exercise 9.5: Train DiT on MNIST

Train the DiT above on MNIST for 10 epochs using cosine schedule and DDPM training. Use class conditioning with 10% label dropout for CFG (use class 10 as the null token).

In [ ]:
# Exercise 9.5 — YOUR CODE HERE

# ========================== END YOUR CODE ==========================

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
torch.manual_seed(42)

dit_model = DiT(embed_dim=128, depth=4, num_heads=4).to(device)
optimizer = torch.optim.Adam(dit_model.parameters(), lr=1e-4)
dataloader = get_mnist_dataloader(batch_size=64)

T_dit = 1000
sched_dit = cosine_schedule(T_dit)
sqrt_abar = sched_dit['sqrt_alphas_cumprod'].to(device)
sqrt_omabar = sched_dit['sqrt_one_minus_alphas_cumprod'].to(device)
null_class = 10
p_uncond = 0.1

loss_history_dit = []
for epoch in range(10):
    epoch_loss = 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        B = images.shape[0]
        # Label dropout
        drop_mask = torch.rand(B, device=device) < p_uncond
        labels = torch.where(drop_mask, torch.full_like(labels, null_class), labels)
        # Forward diffusion
        t_idx = torch.randint(0, T_dit, (B,), device=device)
        noise = torch.randn_like(images)
        x_t = sqrt_abar[t_idx, None, None, None] * images + sqrt_omabar[t_idx, None, None, None] * noise
        noise_pred = dit_model(x_t, t_idx.float() / T_dit, labels)
        loss = F.mse_loss(noise_pred, noise)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(dataloader)
    loss_history_dit.append(avg)
    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1}/10 | Loss: {avg:.4f}")

plot_loss_curve(loss_history_dit, title='DiT Training Loss on MNIST')
print("DiT training complete! Transformers work for diffusion.")

---

## 9.8 — Flow Matching

Flow matching is a simpler alternative to DDPM that learns a **velocity field** transporting noise to data along straight paths.

**Interpolation:** instead of the DDPM noising formula, define a straight-line path:
$$x_t = (1 - t)\,x_0 + t\,\epsilon, \quad t \in [0, 1]$$

**Target velocity:** the derivative along this path is constant:
$$v^*(x_t, t) = \epsilon - x_0$$

**Training:** predict this velocity with an MSE loss:
$$\mathcal{L} = \mathbb{E}_{t, x_0, \epsilon}\left[\|v_\theta(x_t, t) - (\epsilon - x_0)\|^2\right]$$

**Sampling:** integrate the learned velocity from $t=1$ (noise) to $t=0$ (data) using an ODE solver (Euler steps work fine).

### Why flow matching?

- **Straight paths** mean fewer integration steps are needed
- **No noise schedule** to tune
- **Simpler math** with no $\bar\alpha_t$ bookkeeping
- Competitive quality with DDPM at 10-50x fewer sampling steps

### Worked Example: Flow Matching on 2D Point Clouds

In [ ]:
# Generate 2D target distribution (two moons)
def make_moons(n=1000, noise=0.05):
    """Generate two-moons dataset."""
    t = torch.linspace(0, math.pi, n // 2)
    upper = torch.stack([torch.cos(t), torch.sin(t)], dim=1)
    lower = torch.stack([1 - torch.cos(t), 1 - torch.sin(t) - 0.5], dim=1)
    data = torch.cat([upper, lower], dim=0)
    data += noise * torch.randn_like(data)
    return data


class FlowMatchingMLP(nn.Module):
    """Simple MLP that predicts velocity v(x_t, t)."""
    def __init__(self, dim=2, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, dim),
        )

    def forward(self, x, t):
        """(B, dim), (B, 1) -> (B, dim)"""
        if t.dim() == 1:
            t = t.unsqueeze(1)
        return self.net(torch.cat([x, t], dim=1))


# Train flow matching on 2D moons
torch.manual_seed(42)
target_data = make_moons(n=2000, noise=0.05).to(device)
fm_model = FlowMatchingMLP(dim=2, hidden_dim=128).to(device)
optimizer = torch.optim.Adam(fm_model.parameters(), lr=1e-3)

losses_fm = []
for step in range(5000):
    idx = torch.randint(0, len(target_data), (256,))
    x_0 = target_data[idx]                              # (B, 2) data
    x_1 = torch.randn(256, 2, device=device)            # (B, 2) noise
    t = torch.rand(256, 1, device=device)                # (B, 1) in [0, 1]
    x_t = (1 - t) * x_0 + t * x_1                       # (B, 2) interpolation
    target_v = x_1 - x_0                                 # (B, 2) velocity
    pred_v = fm_model(x_t, t)                            # (B, 2)
    loss = F.mse_loss(pred_v, target_v)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if step % 500 == 0:
        losses_fm.append(loss.item())

print(f"Final loss: {losses_fm[-1]:.4f}")

In [ ]:
# Sample from trained flow matching model
@torch.no_grad()
def flow_matching_sample(model, n_samples=500, dim=2, n_steps=50, device=torch.device('cpu')):
    """Sample by integrating learned vector field from t=1 (noise) to t=0 (data)."""
    x = torch.randn(n_samples, dim, device=device)
    dt = 1.0 / n_steps
    for i in range(n_steps):
        t = torch.full((n_samples, 1), 1 - i * dt, device=device)
        x = x - dt * model(x, t)  # Euler step backward
    return x

fm_model.eval()
samples = flow_matching_sample(fm_model, n_samples=1000, n_steps=50, device=device)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(target_data[:, 0].cpu(), target_data[:, 1].cpu(), s=2, alpha=0.5, c='steelblue')
axes[0].set_title('Target Distribution'); axes[0].set_xlim(-2, 3); axes[0].set_ylim(-1.5, 2)

axes[1].scatter(samples[:, 0].cpu(), samples[:, 1].cpu(), s=2, alpha=0.5, c='coral')
axes[1].set_title('Flow Matching Samples'); axes[1].set_xlim(-2, 3); axes[1].set_ylim(-1.5, 2)

# Quiver plot of learned vector field
xx, yy = np.meshgrid(np.linspace(-2, 3, 20), np.linspace(-1.5, 2, 15))
grid = torch.tensor(np.stack([xx.ravel(), yy.ravel()], axis=1), dtype=torch.float32, device=device)
with torch.no_grad():
    vecs = fm_model(grid, torch.full((len(grid), 1), 0.5, device=device)).cpu().numpy()
axes[2].quiver(xx.ravel(), yy.ravel(), -vecs[:, 0], -vecs[:, 1], scale=30, alpha=0.7, color='forestgreen')
axes[2].set_title('Learned Vector Field (t=0.5)'); axes[2].set_xlim(-2, 3); axes[2].set_ylim(-1.5, 2)

plt.suptitle('Flow Matching: 2D Two-Moons', fontsize=14)
plt.tight_layout(); plt.show()

### Exercise 9.6: Flow Matching on MNIST Latents

Apply flow matching to the MNIST latent codes from Section 9.3:
1. Train a `FlowMatchingMLP` with `dim=32` for 30 epochs on `latent_dataset_norm`
2. Sample latents using the ODE solver, then decode through the VAE
3. Compare with DDPM-based latent diffusion from Section 9.3

In [ ]:
# Exercise 9.6 — YOUR CODE HERE

# ========================== END YOUR CODE ==========================

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
torch.manual_seed(42)

fm_latent = FlowMatchingMLP(dim=32, hidden_dim=256).to(device)
optimizer = torch.optim.Adam(fm_latent.parameters(), lr=1e-3)

for epoch in range(30):
    epoch_loss = 0
    for (z_0,) in latent_loader:
        z_0 = z_0.to(device)
        B = z_0.shape[0]
        z_1 = torch.randn_like(z_0)
        t = torch.rand(B, 1, device=device)
        z_t = (1 - t) * z_0 + t * z_1
        target_v = z_1 - z_0
        pred_v = fm_latent(z_t, t)
        loss = F.mse_loss(pred_v, target_v)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/30 | Loss: {epoch_loss/len(latent_loader):.4f}")

fm_latent.eval()
sampled_z = flow_matching_sample(fm_latent, n_samples=16, dim=32, n_steps=50, device=device)
sampled_z = sampled_z * latent_std.to(device) + latent_mean.to(device)
with torch.no_grad():
    fm_images = vae.decode(sampled_z)
show_images(denormalize(fm_images), nrow=8, title='Flow Matching Latent Diffusion Samples')
print("Flow matching: simpler training, comparable results, fewer sampling steps!")

---

## Capstone Exercise: Full Latent Diffusion Pipeline

Bring everything together:

1. **Train a convolutional VAE** on MNIST (latent dim = 32) -- or reuse from Section 9.2
2. **Encode** all training images to latent space
3. **Train a diffusion model** (DDPM or flow matching) on latent codes
4. **Sample:** generate random latent → decode → image
5. **Compare:** pixel-space diffusion vs latent diffusion in quality, speed, and training time

**Bonus:**
- Try v-prediction instead of noise-prediction
- Implement offset noise
- Compare DDPM vs flow matching at different step counts

In [ ]:
# Capstone Exercise — YOUR CODE HERE

# ========================== END YOUR CODE ==========================

In [ ]:
# ✅ SOLUTION — Capstone: Full Latent Diffusion Pipeline
torch.manual_seed(42)

print("=" * 60)
print("CAPSTONE: Latent Diffusion vs Pixel-Space Diffusion")
print("=" * 60)

# --- Part 1: Latent diffusion (already trained above) ---
print("\n--- Part 1: Latent Diffusion Samples ---")
latent_model.eval()
vae.eval()

# DDPM samples
ddpm_z = sample_latent_diffusion(latent_model, schedule, n_samples=8, latent_dim=32, device=device)
ddpm_z = ddpm_z * latent_std.to(device) + latent_mean.to(device)
with torch.no_grad():
    ddpm_imgs = vae.decode(ddpm_z)

# Flow matching samples at different step counts
fm_latent.eval()
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
show_row = lambda imgs, row, label: [
    (axes[row, i].imshow(denormalize(imgs[i]).cpu().squeeze(), cmap='gray'),
     axes[row, i].axis('off')) for i in range(min(8, len(imgs)))
] and axes[row, 0].set_ylabel(label, fontsize=11)

show_row(ddpm_imgs, 0, 'DDPM\n(1000 steps)')

for row, n_steps in [(1, 50), (2, 10)]:
    z = flow_matching_sample(fm_latent, n_samples=8, dim=32, n_steps=n_steps, device=device)
    z = z * latent_std.to(device) + latent_mean.to(device)
    with torch.no_grad():
        imgs = vae.decode(z)
    show_row(imgs, row, f'Flow Match\n({n_steps} steps)')

plt.suptitle('Latent Diffusion: DDPM vs Flow Matching', fontsize=14)
plt.tight_layout(); plt.show()

# --- Part 2: Timing comparison ---
print("\n--- Part 2: Speed Comparison ---")
# Pixel-space model
pixel_model = LatentDiffusionMLP(latent_dim=784, hidden_dim=512).to(device)
opt_p = torch.optim.Adam(pixel_model.parameters(), lr=1e-3)

# Time 5 epochs of each
for name, model, opt, loader in [
    ('Pixel (784d)', pixel_model, opt_p, pixel_loader),
]:
    start = time.time()
    for epoch in range(5):
        for (batch,) in loader:
            batch = batch.to(device)
            B = batch.shape[0]
            t_idx = torch.randint(0, T, (B,), device=device)
            noise = torch.randn_like(batch)
            noisy = sqrt_alpha_bar[t_idx].unsqueeze(1) * batch + \
                    sqrt_one_minus_alpha_bar[t_idx].unsqueeze(1) * noise
            pred = model(noisy, t_idx.float() / T)
            loss = F.mse_loss(pred, noise)
            opt.zero_grad(); loss.backward(); opt.step()
    pixel_time = time.time() - start

latent_model_fresh = LatentDiffusionMLP(latent_dim=32, hidden_dim=256).to(device)
opt_l = torch.optim.Adam(latent_model_fresh.parameters(), lr=1e-3)
start = time.time()
for epoch in range(5):
    for (batch,) in latent_loader:
        batch = batch.to(device)
        B = batch.shape[0]
        t_idx = torch.randint(0, T, (B,), device=device)
        noise = torch.randn_like(batch)
        noisy = sqrt_alpha_bar[t_idx].unsqueeze(1) * batch + \
                sqrt_one_minus_alpha_bar[t_idx].unsqueeze(1) * noise
        pred = latent_model_fresh(noisy, t_idx.float() / T)
        loss = F.mse_loss(pred, noise)
        opt_l.zero_grad(); loss.backward(); opt_l.step()
latent_time = time.time() - start

print(f"Pixel-space (784d): {pixel_time:.1f}s for 5 epochs")
print(f"Latent-space (32d): {latent_time:.1f}s for 5 epochs")
print(f"Speedup: {pixel_time/latent_time:.1f}x")
print("\nLatent diffusion: faster training, comparable quality, and scales to high res!")

---

## Module 9 Summary

**What you learned:**
- Pixel-space diffusion does not scale to high resolutions because compute grows with the number of pixels
- A **VAE** compresses images to a small latent space; the ELBO loss balances reconstruction and regularization
- **Latent diffusion** runs the entire noising/denoising process in latent space, then decodes at the end
- Stable Diffusion = frozen VAE + U-Net denoiser + CLIP text encoder
- Three prediction targets ($\epsilon$, $x_0$, $v$) are interconvertible; $v$-prediction balances loss across timesteps
- Cosine schedules, offset noise, and zero terminal SNR each fix a specific problem with the linear schedule
- **DiT** replaces U-Net with a transformer, using adaLN-Zero for timestep/class conditioning
- **Flow matching** learns straight-line velocity fields, enabling high-quality samples in far fewer steps

**Next:** Module 10 applies everything from Modules 0-9 in timed challenge problems.